In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import math
from pathlib import Path

# Description

Notebook to compare new and current gold files.

This notebook creates spatial difference plots between two NetCDF files, subtracting the "old" from the "new". This comparison should be performed whenever there is a need to update the "gold" references files for the tests.

Once a test fails because of a difference with the gold files, the `output` folder under each `basins` will not be deleted and act as the `new_nc` file in this notebook. 

## Setup
Change the below to match the AWSM tests `basins` folder

In [ ]:
%cd /data/projects/iSnobal-CM/awsm/awsm/tests/basins/

### Test basin

In [ ]:
basin_name = "Lakes"

### NetCDF filename and variable to be checked

In [ ]:
em_variables = [
    "net_rad",
    "sensible_heat",
    "latent_heat",
    "snow_soil",
    "precip_advected",
    "sum_EB",
    "evaporation",
    "snowmelt",
    "SWI",
    "cold_content",
]
snow_variables = [
    "thickness",
    "snow_density",
    "specific_mass",
    "liquid_water",
    "temp_surf",
    "temp_lower",
    "temp_snowcover",
    "thickness_lower",
    "water_saturation",
]
lidar_change = [
    "depth_change",
    "rho_change",
    "swe_change",
]

In [ ]:
filename = "model_lidar_change.nc"
variables = lidar_change

In [ ]:
old_nc = xr.open_dataset(f"{basin_name}/gold_hrrr_update/{filename}").drop_vars("projection")
new_nc_file = list(Path('.').glob(f"{basin_name}/output/*/*/*/*/{filename}"))[0]
new_nc = xr.open_dataset(new_nc_file).drop_vars("projection")

In [ ]:
diff = (new_nc - old_nc)
old_nc.close()
new_nc.close()

## Comparison

In [ ]:
def plot_file(file, variable):
    plots = file[variable].plot(
        x='x', y='y', col="time",
        figsize=(11, 5),
        subplot_kws={"box_aspect": 1},
        cbar_kwargs={"shrink": 0.8}
    )
    
    for ax, meta in zip(plots.axs.flat, plots.name_dicts.flat):
        ax.set_xticklabels([])
        ax.set_yticklabels([])
    
    plots.set_titles("{value}")
    plots.fig.suptitle(f"Variable: {variable}")
   
    return plots

### Areal plots

In [ ]:
for variable in variables:
    plot_file(diff, variable)

### Histogram

In [ ]:
def plot_hist(data, variable):
    num_times = data.sizes["time"]
    num_rows = math.ceil(num_times / 2)
    
    fig, axes = plt.subplots(
        num_rows, 2, 
        figsize=(10, 3 * num_rows), 
        dpi=120,
        layout="constrained"
    )
    axes = axes.flatten()

    fig.suptitle(variable)
    
    for i in range(num_times):
        data[variable].isel(time=i).plot.hist(ax=axes[i], bins=30, color='skyblue')
        axes[i].set_xlabel("")

In [ ]:
for variable in variables:
    plot_hist(diff, variable)